# GPW: Periodic Heat and Movable Stitch Lines

Select a GPU runtime, then Run all. Restart first if an older notebook
already loaded these model classes. Loads the latest main from GitHub;
the resolved commit is saved. No model source is embedded.

Edit STITCH_LEARNING_RATES in the settings cell to test one or several
values. Each value runs all four starts from scratch, with identical data
and seeds. Default 0.003 reproduces the local configuration.

Heat stays horizontal on the left and vertical on the right. Initial
stitches form a vertical, offset vertical, diagonal or horizontal line.
Each submodel starts on its own side; supports are free to cross later.
Each stitch then moves independently; there is no line constraint. Kernels are two
ARD RBFs, and uncertainty includes the full stitched residual R.

This is a posterior-adaptive likelihood-minus-KL objective, not a
fixed-model ELBO guarantee. The learned stitches need not recover the material
boundary. Heating is spatially repeating; the exterior is insulated.
Finite-element heat simulation runs on CPU; GP training uses float64 GPU.
No CPU fallback, solver retries or automatic parameter changes.
Live views include each kernel's x/y lengthscales, output variance and
shared noise SD, from epoch 0 onward. Dashed markers show kernel warm-up.


## 1. GitHub source and dependencies


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/content/gpw_periodic_heat')
if ROOT.exists():
    raise FileExistsError("Checkout already exists; use a fresh Colab runtime")
if any(name == "pwgp" or name.startswith("pwgp.") or name == "experiments"
       or name.startswith("experiments.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading a new source version")
os.environ["GIT_TERMINAL_PROMPT"] = "0"
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, "--single-branch",
                REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import subprocess, sys
_GPW_CELL_RESULT = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "/content/gpw_periodic_heat[heat]"], check=True)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 2. Settings


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import gc, json, shutil, zipfile
from dataclasses import asdict, replace
from datetime import UTC, datetime
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display
from google.colab import files
from experiments.periodic_heat_preview import PreviewConfig
from experiments.stitch_line_preview import LayoutConfig, preview
from experiments.stitch_line_heat import RunConfig, build_model, run
from pwgp.linalg import cholesky_solve

STITCH_LEARNING_RATES = [0.003]  # e.g. [0.003, 0.01, 0.03]
DEVICE = "cuda:0"
from experiments.default_hyperparameters import periodic_heat_defaults
CONFIG = replace(periodic_heat_defaults(device=DEVICE))  # Explicit overrides here.
LAYOUTS = CONFIG.layouts
HEAT = LAYOUTS.heat
if not STITCH_LEARNING_RATES or any(not np.isfinite(v) or v <= 0 for v in STITCH_LEARNING_RATES):
    raise ValueError("STITCH_LEARNING_RATES must contain finite positive values")
if torch.device(DEVICE).type != "cuda" or not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime. This notebook does not fall back to CPU")
print({"GPU": torch.cuda.get_device_name(DEVICE), "torch": torch.__version__,
       "CUDA": torch.version.cuda, "dtype": "float64"})
RUN_ROOT = ROOT / "results/periodic_heat_colab" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
(RUN_ROOT / "source_snapshot.zip").write_bytes(SOURCE_ZIP)
(RUN_ROOT / "source_provenance.json").write_text(json.dumps(SOURCE_PROVENANCE, indent=2))
_GPW_CELL_RESULT = display(pd.DataFrame([{"stitch_learning_rate": v, "layouts": len(LAYOUTS.line_angles),
                      "epochs": CONFIG.epochs, "warmup": CONFIG.warmup_epochs}
                      for v in STITCH_LEARNING_RATES]))
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 3. GPU checks and initial layouts


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
for batch, size in [(2, 28), (500, 1), (1, 56)]:
    matrix = (torch.eye(size, device=DEVICE, dtype=torch.float64) + 0.01).repeat(batch, 1, 1)
    matrix.requires_grad_(True)
    rhs = torch.ones(batch, size, 1, device=DEVICE, dtype=torch.float64)
    answer = cholesky_solve(rhs, torch.linalg.cholesky(matrix))
    torch.testing.assert_close(matrix @ answer, rhs)
    answer.square().mean().backward()
    assert torch.isfinite(matrix.grad).all()
del matrix, rhs, answer
probe_heat = replace(HEAT, supports_per_patch=8, stitches=4, q_rank=2)
probe_config = replace(CONFIG, layouts=replace(LAYOUTS, heat=probe_heat))
for index in range(len(LAYOUTS.line_angles)):
    probe = build_model(probe_config, index)
    x = 0.05 + 0.9 * torch.rand(16, 2, device=DEVICE, dtype=torch.float64)
    loss = -probe.elbo_batch(x, torch.sin(8 * x[:, 0]), len(x))[0]
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters())
    del probe, x, loss
torch.cuda.synchronize(DEVICE)
gc.collect()
torch.cuda.empty_cache()
print("GPU factorization, solves, GPW and independent stitch gradients passed")
starting = preview(LAYOUTS, directory=RUN_ROOT / "initial_layouts")
_GPW_CELL_RESULT = display(Image(filename=str(starting / "initial_layouts.png")))
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 4. Train, visualize and save each run


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
summaries = []
for rate_index, rate in enumerate(STITCH_LEARNING_RATES):
    config = replace(CONFIG, learning_rate_line=rate)
    directory = RUN_ROOT / f"rate_{rate_index}_{rate:g}"
    display(Markdown(f"## Stitch learning rate: {rate:g}"))
    live = display(Markdown("Starting..."), display_id=True)

    def show_progress(folder, epoch, handle=live):
        handle.update(Image(filename=str(folder / "progress.png")))

    def completed(folder, metrics, rate=rate, rate_index=rate_index):
        summaries.append({"stitch_learning_rate": rate, **metrics})
        pd.DataFrame(summaries).to_csv(RUN_ROOT / "comparison.csv", index=False)
        display(pd.DataFrame([summaries[-1]]).T)
        display(Image(filename=str(folder / "kernel_evolution.png")))
        parent = folder.parent
        shared = [parent / name for name in ("data.npz", "config.json", "provenance.json", "metrics.json")]
        archive = RUN_ROOT / f"rate_{rate_index}_{folder.name}.zip"
        with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
            bundle.writestr("source_snapshot.zip", SOURCE_ZIP)
            bundle.writestr("source_provenance.json", json.dumps(SOURCE_PROVENANCE, indent=2))
            for path in shared + sorted(p for p in folder.rglob("*") if p.is_file()):
                bundle.write(path, path.relative_to(parent))
        print("Saved:", archive)
        files.download(str(archive))

    result = run(config, directory=directory, capture_git=False,
                 on_snapshot=show_progress, on_complete=completed)
    for name in ("points.png", "fits.png", "training.png"):
        display(Image(filename=str(result / name)))
_GPW_CELL_RESULT = display(pd.DataFrame(summaries)[["stitch_learning_rate", "layout", "rmse_standardized",
        "nlpd_standardized", "coverage_95", "mean_stitch_displacement"]])
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 5. Download the complete comparison


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
_GPW_CELL_RESULT = files.download(archive)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT
